In [175]:
import numpy as np
import pandas as pd
import sklearn
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model

In [176]:
## load the trained model,scaler pickle onehot

model=load_model('model.h5')

## load the encoder and scaler

with open('geo_encoder.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)
with open('gender_encoding.pkl', 'rb') as file:
    loaded_gender_encoder = pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

In [177]:
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [178]:
# Convert to DataFrame (single row)
input_df = pd.DataFrame([input_data])

print(input_df)

   CreditScore Geography Gender  Age  Tenure  Balance  NumOfProducts  \
0          600    France   Male   40       3    60000              2   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               1            50000  


In [179]:
input_df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [180]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]])


C:\ProgramData\Anaconda3\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [181]:
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))

In [182]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [183]:
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))

In [184]:
input_df['Gender'] 


0    Male
Name: Gender, dtype: object

In [185]:
input_df['Gender']=loaded_gender_encoder.transform(input_df['Gender'])

In [186]:
input_df=input_df.drop('Geography',axis=1)


In [187]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,1,40,3,60000,2,1,1,50000


In [188]:
input_df=pd.concat((input_df,geo_encoded_df),axis=1)

In [189]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [190]:
input_df.columns
input_df.size
#CreditScore	Gender	Age	Tenure	Balance	NumOfProducts	HasCrCard	IsActiveMember	EstimatedSalary	Geography_France	Geography_Germany	Geography_Spain

12

In [191]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
print("input_scaled shape:", input_scaled.shape)

input_scaled shape: (1, 12)


In [192]:
#Churning Prediction
prediction=model.predict(input_scaled)

1/1 [==============================] - 0s 138ms/step


In [193]:
(input_scaled.shape[1],)

(12,)

In [194]:
prediction

array([[0.0651648]], dtype=float32)

In [198]:
if prediction>0.5:
    print("Churning")
else:
    print("Not-Churning")

Not-Churning
